In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q opencv-python-headless ultralytics
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

In [ ]:
import os
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from scipy.ndimage import binary_erosion, distance_transform_edt, label
from tqdm import tqdm
from segment_anything import sam_model_registry
from ultralytics import YOLO

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)

In [ ]:
# 2. Configuração
# Caminhos
BASE_DIR = "/content/drive/MyDrive/IC_TIRADS"
DATA_DIR = f"{BASE_DIR}/data_public/ThyroidDataset/Thyroid Dataset"
TN3K_DIR = f"{DATA_DIR}/tn3k"
TG3K_DIR = f"{DATA_DIR}/tg3k"
CKPT_DIR = f"{BASE_DIR}/checkpoints"
FIG_DIR = f"{BASE_DIR}/figures"
MEDSAM_CKPT_PATH = f"{CKPT_DIR}/medsam_vit_b.pth"
WORK_DIR = "/content/work"  # disco local do Colab (embeddings e dados do YOLO)

DECODER_NODULO = "decoder_trained_tn3k_fast.pth"
DECODER_GLANDULA = "decoder_trained_tg3k_fast.pth"

# Execução
TREINAR_MEDSAM = False   # False: usa os decoders já salvos em CKPT_DIR
TREINAR_YOLO = False     # False: usa os detectores já salvos em CKPT_DIR

# Reprodutibilidade
SEED = 42

# MedSAM
IMG_SIZE = 1024
BOX_JITTER = 10         # folga aleatória (px) da caixa no treino; 0 na avaliação
BATCH_SIZE = 4
NUM_EPOCHS = 20
LR = 1e-4
LIMIAR = 0.5            # binarização da máscara prevista

# YOLO
YOLO_EPOCHS = 30
YOLO_IMGSZ = 640
YOLO_BATCH = 16
YOLO_CONF = 0.25

for d in [CKPT_DIR, FIG_DIR, WORK_DIR]:
    os.makedirs(d, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# 3. Dados e splits
def indexar_pares(img_dir, mask_dir, **info):
    """Lista pares imagem/máscara com o mesmo nome de arquivo."""
    img_dir, mask_dir = Path(img_dir), Path(mask_dir)
    if not img_dir.exists():
        print(f"[aviso] pasta não encontrada: {img_dir}")
        return []
    return [
        {"image_path": str(p), "mask_path": str(mask_dir / p.name), **info}
        for p in sorted(img_dir.glob("*"))
        if (mask_dir / p.name).exists()
    ]


df_tn3k = pd.DataFrame(
    indexar_pares(f"{TN3K_DIR}/trainval-image", f"{TN3K_DIR}/trainval-mask", source="tn3k", split_source="train")
    + indexar_pares(f"{TN3K_DIR}/test-image", f"{TN3K_DIR}/test-mask", source="tn3k", split_source="test")
)
df_tg3k = pd.DataFrame(
    indexar_pares(f"{TG3K_DIR}/thyroid-image", f"{TG3K_DIR}/thyroid-mask", source="tg3k", split_source="train")
)
print("TN3K:", len(df_tn3k), "| TG3K:", len(df_tg3k))

In [ ]:
# TN3K
df_tn3k_train, df_tn3k_val = train_test_split(
    df_tn3k[df_tn3k["split_source"] == "train"], test_size=0.15, random_state=SEED
)
df_tn3k_test = df_tn3k[df_tn3k["split_source"] == "test"]

# TG3K
df_tg3k_train, df_tg3k_temp = train_test_split(df_tg3k, test_size=0.30, random_state=SEED)
df_tg3k_val, df_tg3k_test = train_test_split(df_tg3k_temp, test_size=0.50, random_state=SEED)

print(f"TN3K  treino {len(df_tn3k_train)} | val {len(df_tn3k_val)} | teste {len(df_tn3k_test)}")
print(f"TG3K  treino {len(df_tg3k_train)} | val {len(df_tg3k_val)} | teste {len(df_tg3k_test)}")

In [ ]:
# 4. Datasets PyTorch
def carregar_mascara(mask_path, img_size=IMG_SIZE):
    mask = (cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) > 0).astype(np.uint8)
    return cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)


def caixa_da_mascara(mask, img_size=IMG_SIZE, jitter=0):
    """Bounding box [x0, y0, x1, y1] da máscara, com folga aleatória opcional."""
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return np.array([0, 0, img_size, img_size])
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    if jitter > 0:
        x0 = max(0, x0 - np.random.randint(0, jitter))
        y0 = max(0, y0 - np.random.randint(0, jitter))
        x1 = min(img_size, x1 + np.random.randint(0, jitter))
        y1 = min(img_size, y1 + np.random.randint(0, jitter))
    return np.array([x0, y0, x1, y1])


def nome_base(path):
    return os.path.basename(path).split('.')[0]


class ThyroidSAMDataset(Dataset):
    def __init__(self, df, img_size=IMG_SIZE, box_jitter=BOX_JITTER):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.box_jitter = box_jitter

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.cvtColor(cv2.imread(row["image_path"]), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))
        mask = carregar_mascara(row["mask_path"], self.img_size)
        box = caixa_da_mascara(mask, self.img_size, self.box_jitter)

        img_t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        return img_t, torch.from_numpy(box).float(), torch.from_numpy(mask).float()


class ThyroidEmbeddingDataset(Dataset):
    def __init__(self, df, emb_dir, img_size=IMG_SIZE, box_jitter=BOX_JITTER):
        self.df = df.reset_index(drop=True)
        self.emb_dir = emb_dir
        self.img_size = img_size
        self.box_jitter = box_jitter

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mask = carregar_mascara(row["mask_path"], self.img_size)
        box = caixa_da_mascara(mask, self.img_size, self.box_jitter)
        emb = torch.load(f"{self.emb_dir}/{nome_base(row['image_path'])}.pt")
        return emb, torch.from_numpy(box).float(), torch.from_numpy(mask).float()

In [ ]:
# 5. Modelo MedSAM
medsam_model = sam_model_registry["vit_b"](checkpoint=None)
medsam_model.load_state_dict(torch.load(MEDSAM_CKPT_PATH, map_location=device))
medsam_model = medsam_model.to(device)

for p in medsam_model.image_encoder.parameters():
    p.requires_grad = False
for p in medsam_model.prompt_encoder.parameters():
    p.requires_grad = False
for p in medsam_model.mask_decoder.parameters():
    p.requires_grad = True

treinaveis = sum(p.numel() for p in medsam_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in medsam_model.parameters())
print(f"Parâmetros treináveis: {treinaveis:,} / {total:,} ({100 * treinaveis / total:.1f}%)")

In [ ]:
def resetar_decoder(model):
    """Restaura o mask decoder original do MedSAM (antes do fine-tuning)."""
    sd = torch.load(MEDSAM_CKPT_PATH, map_location=device)
    dec = {k[len("mask_decoder."):]: v for k, v in sd.items() if k.startswith("mask_decoder.")}
    model.mask_decoder.load_state_dict(dec)


def carregar_decoder(model, nome_checkpoint):
    """Carrega um decoder treinado salvo em CKPT_DIR."""
    ckpt = torch.load(f"{CKPT_DIR}/{nome_checkpoint}", map_location=device)
    state = ckpt["decoder_state"] if isinstance(ckpt, dict) and "decoder_state" in ckpt else ckpt
    model.mask_decoder.load_state_dict(state)


@torch.no_grad()
def prever_mascara(model, img_t, box_t):
    """Segmenta uma imagem (3×1024×1024) a partir de uma caixa [x0, y0, x1, y1]."""
    model.eval()
    emb = model.image_encoder(img_t.unsqueeze(0).to(device))
    sparse, dense = model.prompt_encoder(points=None, boxes=box_t.unsqueeze(0).to(device), masks=None)
    low_res, _ = model.mask_decoder(
        image_embeddings=emb,
        image_pe=model.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse,
        dense_prompt_embeddings=dense,
        multimask_output=False,
    )
    pred = F.interpolate(low_res, size=img_t.shape[-2:], mode="bilinear", align_corners=False)
    return (torch.sigmoid(pred).squeeze().cpu().numpy() > LIMIAR).astype(np.uint8)

In [ ]:
# 6. Treino do decoder
@torch.no_grad()
def precalcular_embeddings(df, model, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    model.eval()
    ds = ThyroidSAMDataset(df, box_jitter=0)
    for i in tqdm(range(len(ds)), desc="Embeddings"):
        save_path = f"{save_dir}/{nome_base(df.iloc[i]['image_path'])}.pt"
        if os.path.exists(save_path):  # permite retomar se o Colab cair
            continue
        img, _, _ = ds[i]
        emb = model.image_encoder(img.unsqueeze(0).to(device)).squeeze(0).cpu()
        torch.save(emb, save_path)


def dice_loss(pred, target, eps=1e-6):
    pred = torch.sigmoid(pred)
    inter = (pred * target).sum(dim=(1, 2))
    union = pred.sum(dim=(1, 2)) + target.sum(dim=(1, 2))
    return 1 - ((2 * inter + eps) / (union + eps)).mean()


def treinar_uma_epoca(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for emb, box, mask in loader:
        emb, box, mask = emb.to(device), box.to(device), mask.to(device)
        loss = 0.0
        for i in range(emb.shape[0]):
            with torch.no_grad():
                sparse, dense = model.prompt_encoder(points=None, boxes=box[i:i + 1], masks=None)
            low_res, _ = model.mask_decoder(
                image_embeddings=emb[i:i + 1],
                image_pe=model.prompt_encoder.get_dense_pe(),
                sparse_prompt_embeddings=sparse,
                dense_prompt_embeddings=dense,
                multimask_output=False,
            )
            pred = F.interpolate(low_res, size=mask.shape[-2:], mode="bilinear", align_corners=False).squeeze(1)
            alvo = mask[i:i + 1]
            loss += dice_loss(pred, alvo) + F.binary_cross_entropy_with_logits(pred, alvo)
        loss = loss / emb.shape[0]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def treinar_decoder(model, df_train, emb_dir, nome_checkpoint, rotulo):
    precalcular_embeddings(df_train, model, emb_dir)
    loader = DataLoader(ThyroidEmbeddingDataset(df_train, emb_dir),
                        batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    optimizer = torch.optim.AdamW(model.mask_decoder.parameters(), lr=LR)
    for epoch in range(NUM_EPOCHS):
        loss = treinar_uma_epoca(model, loader, optimizer)
        print(f"[{rotulo}] época {epoch + 1}/{NUM_EPOCHS} - loss: {loss:.4f}")
        torch.save({
            "epoch": epoch,
            "decoder_state": model.mask_decoder.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "loss": loss,
        }, f"{CKPT_DIR}/{nome_checkpoint}")

In [ ]:
# 6.1 Nódulo (TN3K)
if TREINAR_MEDSAM:
    resetar_decoder(medsam_model)
    treinar_decoder(medsam_model, df_tn3k_train, f"{WORK_DIR}/embeddings_tn3k_train", DECODER_NODULO, "nódulo")

In [ ]:
# 6.2 Glândula (TG3K)
if TREINAR_MEDSAM:
    resetar_decoder(medsam_model)
    treinar_decoder(medsam_model, df_tg3k_train, f"{WORK_DIR}/embeddings_tg3k_train", DECODER_GLANDULA, "glândula")

In [ ]:
# 7. Métricas
def _borda(m):
    m = m.astype(bool)
    return m & ~binary_erosion(m)


def distancias_hausdorff(pred, gt):
    """Retorna (HD95, HD máxima) simétricas entre os contornos, em pixels."""
    pb, gb = _borda(pred), _borda(gt)
    if not pb.any() or not gb.any():
        return np.nan, np.nan
    d = np.concatenate([distance_transform_edt(~gb)[pb], distance_transform_edt(~pb)[gb]])
    return np.percentile(d, 95), d.max()


def calcular_metricas(pred, gt, eps=1e-6):
    tp = int(((pred == 1) & (gt == 1)).sum())
    fp = int(((pred == 1) & (gt == 0)).sum())
    fn = int(((pred == 0) & (gt == 1)).sum())
    tn = int(((pred == 0) & (gt == 0)).sum())
    hd95, hd_max = distancias_hausdorff(pred, gt)
    return {
        "dice": 2 * tp / (2 * tp + fp + fn + eps),
        "iou": tp / (tp + fp + fn + eps),
        "precision": tp / (tp + fp + eps),
        "recall_sensitivity": tp / (tp + fn + eps),
        "specificity": tn / (tn + fp + eps),
        "pixel_accuracy": (tp + tn) / (tp + tn + fp + fn + eps),
        "hd95": hd95,
        "hd_max": hd_max,
    }


def avaliar(model, df, boxes=None):
    """Avalia no df. Sem `boxes`: caixas do gabarito (assistido). Com `boxes`: caixas do detector."""
    ds = ThyroidSAMDataset(df, box_jitter=0)
    linhas = []
    for i in tqdm(range(len(ds)), desc="Avaliação"):
        img_t, box_t, mask_t = ds[i]
        path = df.iloc[i]["image_path"]
        if boxes is not None:
            box_t = torch.from_numpy(boxes[path]).float()
        pred = prever_mascara(model, img_t, box_t)
        linhas.append({"image_path": path, **calcular_metricas(pred, mask_t.numpy().astype(np.uint8))})
    return pd.DataFrame(linhas)


def resumir(df_metricas, titulo):
    print(f"=== {titulo} (n={len(df_metricas)}) ===")
    print(df_metricas.drop(columns="image_path").agg(["mean", "std", "median"]).round(4), "\n")

In [ ]:
# 8. Avaliação assistida (caixa do gabarito, sem folga)
carregar_decoder(medsam_model, DECODER_NODULO)
df_metrics_nodulo = avaliar(medsam_model, df_tn3k_test)
resumir(df_metrics_nodulo, "Assistido - nódulo (TN3K)")
df_metrics_nodulo.to_csv(f"{BASE_DIR}/metrics_nodulo_full.csv", index=False)

carregar_decoder(medsam_model, DECODER_GLANDULA)
df_metrics_gland = avaliar(medsam_model, df_tg3k_test)
resumir(df_metrics_gland, "Assistido - glândula (TG3K)")
df_metrics_gland.to_csv(f"{BASE_DIR}/metrics_gland_full.csv", index=False)

In [ ]:
print("Piores casos de HD95 - glândula:")
print(df_metrics_gland.nlargest(5, "hd95")[["image_path", "dice", "hd95"]])

In [ ]:
# 9. Detector YOLO
def mascara_para_bbox_yolo(mask_path):
    """Bounding box da máscara no formato YOLO (cx, cy, w, h normalizados)."""
    mask = (cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) > 0).astype(np.uint8)
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return None
    h, w = mask.shape
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    return (x0 + x1) / 2 / w, (y0 + y1) / 2 / h, (x1 - x0) / w, (y1 - y0) / h


def preparar_dataset_yolo(splits, output_dir, nome_classe):
    for split_name, df in splits.items():
        img_dir = Path(output_dir) / "images" / split_name
        lbl_dir = Path(output_dir) / "labels" / split_name
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)
        for _, row in df.iterrows():
            bbox = mascara_para_bbox_yolo(row["mask_path"])
            if bbox is None:
                continue
            nome = Path(row["image_path"]).stem
            cv2.imwrite(str(img_dir / f"{nome}.jpg"), cv2.imread(row["image_path"]))
            with open(lbl_dir / f"{nome}.txt", "w") as f:
                f.write("0 " + " ".join(f"{v:.6f}" for v in bbox) + "\n")
        print(f"{split_name}: {len(list(img_dir.glob('*.jpg')))} imagens")

    yaml_path = f"{output_dir}/dataset.yaml"
    with open(yaml_path, "w") as f:
        f.write(f"path: {output_dir}\ntrain: images/train\nval: images/val\ntest: images/test\n"
                f"names:\n  0: {nome_classe}\n")
    return yaml_path


def treinar_yolo(yaml_path, nome_execucao):
    YOLO("yolov8n.pt").train(
        data=yaml_path, epochs=YOLO_EPOCHS, imgsz=YOLO_IMGSZ, batch=YOLO_BATCH,
        project=CKPT_DIR, name=nome_execucao, exist_ok=True,
    )

In [ ]:
if TREINAR_YOLO:
    yaml_nodulo = preparar_dataset_yolo(
        {"train": df_tn3k_train, "val": df_tn3k_val, "test": df_tn3k_test}, f"{WORK_DIR}/yolo_tn3k", "nodulo")
    treinar_yolo(yaml_nodulo, "yolo_nodulo_detector")

    yaml_gland = preparar_dataset_yolo(
        {"train": df_tg3k_train, "val": df_tg3k_val, "test": df_tg3k_test}, f"{WORK_DIR}/yolo_tg3k", "glandula")
    treinar_yolo(yaml_gland, "yolo_glandula_detector")

yolo_nodulo = YOLO(f"{CKPT_DIR}/yolo_nodulo_detector/weights/best.pt")
yolo_gland = YOLO(f"{CKPT_DIR}/yolo_glandula_detector/weights/best.pt")

In [ ]:
# 10. Avaliação automática (YOLO + MedSAM)
def detectar_caixas(df, yolo_model, img_size=IMG_SIZE, conf=YOLO_CONF):
    """Retorna {image_path: caixa em 1024×1024} e o conjunto de imagens com detecção."""
    caixas, detectadas = {}, set()
    for _, row in df.iterrows():
        h, w = cv2.imread(row["image_path"]).shape[:2]
        res = yolo_model.predict(row["image_path"], conf=conf, verbose=False)[0]
        if len(res.boxes) == 0:
            caixas[row["image_path"]] = np.array([0, 0, img_size, img_size])
            continue
        x0, y0, x1, y1 = res.boxes.xyxy[res.boxes.conf.argmax().item()].cpu().numpy()
        caixas[row["image_path"]] = np.array([x0 * img_size / w, y0 * img_size / h,
                                              x1 * img_size / w, y1 * img_size / h])
        detectadas.add(row["image_path"])
    return caixas, detectadas


def avaliar_automatico(df, yolo_model, nome_decoder, rotulo, arquivo_csv):
    caixas, detectadas = detectar_caixas(df, yolo_model)
    carregar_decoder(medsam_model, nome_decoder)
    df_m = avaliar(medsam_model, df, boxes=caixas)
    df_m["detectado"] = df_m["image_path"].isin(detectadas)
    df_m.to_csv(f"{BASE_DIR}/{arquivo_csv}", index=False)

    n, n_det = len(df_m), int(df_m["detectado"].sum())
    print(f"[{rotulo}] Taxa de detecção: {n_det}/{n} ({100 * n_det / n:.1f}%)\n")
    resumir(df_m.drop(columns="detectado"), f"Automático - {rotulo} - todas")
    resumir(df_m[df_m["detectado"]].drop(columns="detectado"), f"Automático - {rotulo} - com detecção")
    if n_det < n:
        resumir(df_m[~df_m["detectado"]].drop(columns="detectado"), f"Automático - {rotulo} - sem detecção (fallback)")
    return df_m

In [ ]:
df_metrics_automatico = avaliar_automatico(
    df_tn3k_test, yolo_nodulo, DECODER_NODULO, "nódulo", "metrics_nodulo_automatico.csv")

In [ ]:
df_metrics_automatico_gland = avaliar_automatico(
    df_tg3k_test, yolo_gland, DECODER_GLANDULA, "glândula", "metrics_gland_automatico.csv")

In [ ]:
# 11. Artefato de anotação no TG3K
def tem_artefato(mask_path, tamanho_canto=100, area_min=10, area_max=2000):
    mask = (cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) > 0).astype(np.uint8)
    rotulos, n = label(mask)
    if n <= 1:
        return False
    areas = [(rotulos == i).sum() for i in range(1, n + 1)]
    maior = int(np.argmax(areas)) + 1
    for i in range(1, n + 1):
        if i == maior or not (area_min <= areas[i - 1] <= area_max):
            continue
        ys, xs = np.where(rotulos == i)
        if ys.max() < tamanho_canto and xs.max() < tamanho_canto:
            return True
    return False


artefato_teste = df_tg3k_test["mask_path"].apply(tem_artefato)
artefato_total = df_tg3k["mask_path"].apply(tem_artefato)
print(f"Teste: {artefato_teste.sum()} de {len(df_tg3k_test)} ({100 * artefato_teste.mean():.1f}%)")
print(f"TG3K completo: {artefato_total.sum()} de {len(df_tg3k)} ({100 * artefato_total.mean():.1f}%)")

In [ ]:
# Inspeção visual do canto superior esquerdo em casos marcados
casos = list(np.where(artefato_teste.values)[0])
for idx in random.Random(SEED).sample(casos, 5):
    img = cv2.imread(df_tg3k_test.iloc[idx]["image_path"])
    plt.figure(figsize=(3, 3))
    plt.imshow(cv2.cvtColor(img[:100, :100], cv2.COLOR_BGR2RGB))
    plt.title(f"Índice {idx}")
    plt.axis("off")
    plt.show()